# CNNの学習とGrad-CAM++による可視化（Google Colab）

手書き数字の **MNIST** を用いてCNNを学習し、未知画像に対して予測結果とGrad-CAM++を表示します。

**授業で変更する場所**：第2節のフィルタ数、第3節のエポック数、第6節の画像番号 `INDEXES`。
学習データはKerasから取得し、未知画像はGitHub上の別リポジトリから取得します（画像はこのノートブックに同梱しません）。

**注意**：Grad-CAM++は分類スコアに関連する領域を示すもので、モデルが実際にその画素だけを使ったと証明するものではありません。

## 1．MNISTの読み込み

In [ ]:
#@title MNISTとライブラリの準備
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import subprocess

# 実行するたびにモデル学習の結果が変わりにくいようにする
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# KerasのMNIST：黒い背景・白い数字、28×28、0～9の10クラス
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
X_train = (X_train.astype("float32") / 255.0)[..., None]
X_test  = (X_test.astype("float32") / 255.0)[..., None]
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test  = tf.keras.utils.to_categorical(y_test, 10)
print("学習画像:", X_train.shape, "独立テスト画像:", X_test.shape)


## 2．畳み込み層のフィルタ数を変更

In [ ]:
#@title CNNの畳み込み層の構成
# 2～4層目を0にすると、その層を省略します（1層目は必須）。
UNITS_COV1 = 32 #@param {type:"slider", min:32, max:128, step:32}
UNITS_COV2 = 0  #@param {type:"slider", min:0, max:128, step:32}
UNITS_COV3 = 0  #@param {type:"slider", min:0, max:128, step:32}
UNITS_COV4 = 0  #@param {type:"slider", min:0, max:128, step:32}


## 3．CNNの学習

最適化手法はSGD、損失関数はCategorical Crossentropy、バッチサイズは32です。
学習画像の一部（10%）を学習中の検証に使い、**MNISTのテスト集合は最後の評価まで学習・検証に使いません**。畳み込み層は3×3、stride=1、padding=sameです。プーリング層は用いません。

In [ ]:
#@title 学習モデルのトレーニング
EPOCHS = 1 #@param {type:"slider", min:1, max:20, step:1}

inputs = tf.keras.Input(shape=(28, 28, 1), name="input")
x = inputs
# 0以外の層だけ追加する。学生は上のスライダーで層数とフィルタ数を変更できます。
filters = [UNITS_COV1, UNITS_COV2, UNITS_COV3, UNITS_COV4]
active_layers = []
for i, n_filter in enumerate(filters, start=1):
    if n_filter > 0:
        layer_name = f"conv{i}"
        x = tf.keras.layers.Conv2D(n_filter, (3, 3), padding="same",
                                   activation="relu", name=layer_name)(x)
        active_layers.append(layer_name)

x = tf.keras.layers.Flatten(name="flatten")(x)
x = tf.keras.layers.Dense(64, activation="relu", name="fc")(x)
logits = tf.keras.layers.Dense(10, name="logits")(x)
outputs = tf.keras.layers.Activation("softmax", name="softmax")(logits)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer="SGD", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

history = model.fit(X_train, y_train, validation_split=0.1,
                    epochs=EPOCHS, batch_size=32, verbose=1)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"MNIST独立テスト正解率: {test_acc:.3f} / 損失: {test_loss:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for j, metric in enumerate(["accuracy", "loss"]):
    ax[j].plot(history.history[metric], marker="o", label="Train")
    ax[j].plot(history.history["val_" + metric], marker="s", label="Validation")
    ax[j].set_title(metric.capitalize())
    ax[j].set_xlabel("Epoch")
    ax[j].grid(True)
    ax[j].legend()
plt.tight_layout()
plt.show()


## 4．GitHubから未知の数字画像を取得

画像は [TakaNori999/Num_sorce_mnist](https://github.com/TakaNori999/Num_sorce_mnist) の `num_src` に保存された、著者が独自に作成した数字0～9の各3枚（計30枚）です。**取得元コミットを固定**し、授業時に同じ画像を使用できるようにしています。画像を再利用・再配布する際の条件は、画像リポジトリに示される許諾条件に従ってください。

In [ ]:
#@title 未知画像の取得と一覧表示
DATA_URL = "https://github.com/TakaNori999/Num_sorce_mnist.git"
DATA_REVISION = "c41ab788a669ad03b58b1e387b69647faa699eaf"
REPO_DIR = Path("/content/Num_sorce_mnist_gradcampp")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", DATA_URL, str(REPO_DIR)], check=True)
if not (REPO_DIR / ".git").is_dir():
    raise RuntimeError("取得先に別のフォルダがあります。REPO_DIRを変更してください。")
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", DATA_REVISION], check=True)

data_dir = REPO_DIR / "num_src"
img_paths = sorted(data_dir.glob("*.png"))
if not img_paths:
    raise FileNotFoundError(f"数字画像が見つかりません: {data_dir}")

imgs_ext_show, X_ext_model, y_ext = [], [], []
for path in img_paths:
    # ファイル名例: 3_01.png （先頭の数字が正解ラベル）
    label = int(path.name.split("_")[0])
    if not 0 <= label <= 9:
        raise ValueError(f"正解ラベルが0～9ではありません: {path.name}")
    with Image.open(path) as img:
        original = np.asarray(img.convert("L").resize((28, 28)), dtype=np.float32) / 255.0
    imgs_ext_show.append(original)     # 白地に黒い数字（表示用）
    X_ext_model.append(1.0 - original) # 黒地に白い数字（MNISTと同じ向き）
    y_ext.append(label)

X_ext_model = np.asarray(X_ext_model, dtype=np.float32)[..., None]
y_ext = np.asarray(y_ext, dtype=np.int32)
print("外部画像数:", len(img_paths), "正解ラベル:", np.bincount(y_ext, minlength=10).tolist())
external_loss, external_acc = model.evaluate(X_ext_model,
                           tf.keras.utils.to_categorical(y_ext, 10), verbose=0)
print(f"外部画像の正解率: {external_acc:.3f} （{len(y_ext)}枚）")

cols = 6
rows = int(np.ceil(len(img_paths) / cols))
plt.figure(figsize=(cols * 1.8, rows * 1.8))
for i, img in enumerate(imgs_ext_show):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(img, cmap="gray", vmin=0, vmax=1)
    plt.title(f"#{i} / {y_ext[i]}", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()


## 5．Grad-CAM++の計算（2階・3階微分を反映）

対象となる畳み込み層の特徴マップを $A$、クラス $c$ のSoftmax前のスコアを $S^c$ とします。Grad-CAM++の原論文では、$Y^c=\exp(S^c)$ を用いると、後続の層が線形演算とReLUから成る場合に以下が成立します（ReLUの折れ点を除く）。

$$
\frac{\partial Y^c}{\partial A}=Y^c g,\quad
\frac{\partial^2 Y^c}{\partial A^2}=Y^c g^2,\quad
\frac{\partial^3 Y^c}{\partial A^3}=Y^c g^3,\qquad
 g=\frac{\partial S^c}{\partial A}.
$$

**ここでは2階・3階微分を解析的に正確に計算**し、Grad-CAM++の係数に代入します。全画素について3階の自動微分を繰り返す必要はありません。指数関数の数値オーバーフローを避けるため、$\exp(S^c-\mathrm{stop\_gradient}(S^c))$ として計算します。これは対象画像における正の定数倍の変更で、正規化後のマップは変わりません。

原論文：Chattopadhyay et al., *Grad-CAM++*, WACV 2018, https://doi.org/10.1109/WACV.2018.00097。なお、ここでの「正確」とは**定義した指数スコアに関する微分**が解析的に正しいという意味であり、ヒートマップ自体の忠実性を保証するものではありません。

In [ ]:
#@title Grad-CAM++関数（解析的な2階・3階微分）
def gradcampp_for_layer(model, img4d, layer_name, class_idx=None, eps=1e-8):
    # Grad-CAM++ (WACV 2018): Y^c = exp(S^c), S^c はSoftmax前のlogit。
    # この教材の分類部はFlatten → Dense(ReLU) → Dense(線形)なので、
    # 対象特徴マップAからlogitへの写像はReLUの折れ点以外で局所的に線形です。
    # よって Y の2階・3階微分は近似でなく、1階勾配gから厳密に計算できます。
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.get_layer("logits").output]
    )
    with tf.GradientTape() as tape:
        fmap, logits = grad_model(img4d, training=False)
        if class_idx is None:
            class_idx = int(tf.argmax(logits[0]).numpy())
        score = logits[:, class_idx]              # S^c
    grad_score = tape.gradient(score, fmap)       # g = dS^c/dA
    if grad_score is None:
        raise RuntimeError("勾配を計算できません。対象の畳み込み層を確認してください。")

    fmap = tf.cast(fmap[0], tf.float32)           # (height, width, channel)
    g = tf.cast(grad_score[0], tf.float32)

    # Y = exp(S^c - stop_gradient(S^c))：指数の引数の値は0、勾配はS^cに依存。
    # 指数スコアを定数倍するだけなので、alphaと正規化後CAMは変化しません。
    y = tf.exp(score - tf.stop_gradient(score))[0]  # 数値的には1
    d1 = y * g                                  # 正確な1階微分 dY/dA
    d2 = y * tf.square(g)                       # 正確な2階微分 d²Y/dA²
    d3 = y * tf.pow(g, 3)                       # 正確な3階微分 d³Y/dA³

    # alpha_ijk = d2_ijk / (2*d2_ijk + (sum_ab A_abk)*d3_ijk)
    sum_fmap = tf.reduce_sum(fmap, axis=(0, 1), keepdims=True)
    # 上式をd2で割った等価な形。高階微分 d3/d2 を実際の重み計算に使う。
    # tf.math.divide_no_nanにより d2=0 の位置での0除算を防ぐ。
    ratio_3_to_2 = tf.math.divide_no_nan(d3, d2)
    denom_reduced = 2.0 + sum_fmap * ratio_3_to_2
    safe_denom = tf.where(
        tf.abs(denom_reduced) < eps,
        tf.where(denom_reduced < 0.0, -eps, eps),
        denom_reduced
    )
    alpha = tf.where(tf.abs(g) > eps, 1.0 / safe_denom, tf.zeros_like(g))
    alpha = tf.where(tf.math.is_finite(alpha), alpha, tf.zeros_like(alpha))

    # Grad-CAM++では正の1階微分を用いてチャネル重みを計算する。
    weights = tf.reduce_sum(alpha * tf.nn.relu(d1), axis=(0, 1))
    cam = tf.nn.relu(tf.reduce_sum(fmap * weights, axis=-1))
    cam = tf.image.resize(cam[..., None], (28, 28), method="bilinear")[:, :, 0]
    cam = tf.nn.relu(cam)
    maximum = float(tf.reduce_max(cam).numpy())
    cam = cam / maximum if maximum > eps else tf.zeros_like(cam)
    return cam.numpy(), int(class_idx)


## 6．画像を選択して予測・可視化

In [ ]:
#@title 未知画像の予測とGrad-CAM++
# 第4節に表示された画像番号を指定します。例: [0] / [0, 5, 10]
INDEXES = [0] #@param {type:"raw"}

# 最後に追加された畳み込み層でGrad-CAM++を計算します
TARGET_LAYER = active_layers[-1]

if not INDEXES:
    print("INDEXESに画像番号を1つ以上入力してください。")
else:
    fig, axes = plt.subplots(len(INDEXES), 3,
                             figsize=(10, 2.8 * len(INDEXES)), squeeze=False)
    for r, idx in enumerate(INDEXES):
        if not isinstance(idx, int) or not (0 <= idx < len(img_paths)):
            print(f"画像番号 {idx} は範囲外のためスキップしました（0～{len(img_paths)-1}）。")
            for ax in axes[r]:
                ax.axis("off")
            continue

        img4d = X_ext_model[idx:idx + 1]
        prob = model(img4d, training=False).numpy()[0]
        pred = int(np.argmax(prob))
        heatmap, _ = gradcampp_for_layer(model, img4d, TARGET_LAYER, pred)
        axes[r, 0].imshow(imgs_ext_show[idx], cmap="gray", vmin=0, vmax=1)
        axes[r, 0].set_title(f"#{idx} true={y_ext[idx]}")
        axes[r, 1].imshow(imgs_ext_show[idx], cmap="gray", vmin=0, vmax=1)
        axes[r, 1].imshow(heatmap, cmap="jet", vmin=0, vmax=1, alpha=0.55)
        axes[r, 1].set_title(f"Grad-CAM++: pred={pred}, p={prob[pred]:.3f}")
        axes[r, 2].imshow(heatmap, cmap="jet", vmin=0, vmax=1)
        axes[r, 2].set_title("Heatmap (0–1)")
        for ax in axes[r]:
            ax.axis("off")
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    print("説明対象層:", TARGET_LAYER)
